# PG-CM Neural Retrieval Experiment

Run the cells in order on a Kaggle GPU notebook. Stop immediately if a test or audit cell fails. The 2,000-trace fixture is fixed to SHA-256 `c7fe0e27b17b963d4499fc08f441b8de9776196d455e616db5c6cb0dbffc1c23`.


## 1. Extract the attached package

Attach `PGCM_Neural_Stage.zip` as a Kaggle input before running this cell.


In [ ]:
from pathlib import Path
import shutil, zipfile

candidates = list(Path('/kaggle/input').rglob('PGCM_Neural_Stage.zip'))
assert len(candidates) == 1, f'Expected one PGCM_Neural_Stage.zip, found: {candidates}'
ROOT = Path('/kaggle/working/pgcm_neural_stage')
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(candidates[0]) as archive:
    archive.extractall(ROOT)
assert (ROOT / 'fixtures.jsonl').is_file()
print(ROOT)


## 2. Install dependencies and run all tests


In [ ]:
%cd /kaggle/working/pgcm_neural_stage
!python -m pip install -q -e pgcm_experiments "sentence-transformers>=3,<6" "huggingface-hub>=0.25,<2"
!python -m unittest discover -s pgcm_experiments/tests -v


In [ ]:
import hashlib, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator in Notebook settings'
fixture_hash = hashlib.sha256(Path('fixtures.jsonl').read_bytes()).hexdigest()
assert fixture_hash == 'c7fe0e27b17b963d4499fc08f441b8de9776196d455e616db5c6cb0dbffc1c23'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'fixture_sha256': fixture_hash})


## 3. Smoke test

This uses a separate directory and does not contaminate the full run.


In [ ]:
!python pgcm_experiments/scripts/build_hf_embeddings.py \
  --dataset fixtures.jsonl --out outputs/embeddings_smoke \
  --model BAAI/bge-small-en-v1.5 --device cuda:0 --batch-size 128 --max-traces 25


## 4. Full 2,000-trace embedding run

This cell is resumable after an interruption. Re-run it with the same arguments.


In [ ]:
!python pgcm_experiments/scripts/build_hf_embeddings.py \
  --dataset fixtures.jsonl --out outputs/embeddings_bge \
  --model BAAI/bge-small-en-v1.5 --device cuda:0 --batch-size 128


## 5. Validate embeddings before retrieval


In [ ]:
!python pgcm_experiments/scripts/verify_neural_stage.py \
  --dataset fixtures.jsonl --vectors-dir outputs/embeddings_bge \
  --out outputs/embedding_audit.json


## 6. Replay the matched retrieval experiment


In [ ]:
!python -m pgcm retrieval \
  --dataset fixtures.jsonl --vectors-dir outputs/embeddings_bge \
  --out outputs/neural_retrieval --bootstrap 10000


## 7. Audit and display the measured results


In [ ]:
!python pgcm_experiments/scripts/summarize_neural_retrieval.py \
  --neural outputs/neural_retrieval/retrieval.json \
  --hash hash_control_retrieval.json --out outputs/neural_summary
from pathlib import Path
print(Path('outputs/neural_summary/NEURAL_RETRIEVAL_RESULTS.md').read_text())


## 8. Export


In [ ]:
import shutil
result_zip = shutil.make_archive('/kaggle/working/PGCM_Neural_Results', 'zip', 'outputs')
print(f'Download from the Kaggle Output panel: {result_zip}')
